# 02 — Data Cleaning and the Model

This notebook prepares the data and writes down the model: the objective and the seven rules.

It runs on its own. It reads nothing from other notebooks and it saves nothing. Notebooks 03, 04 and
05 repeat the same preparation, so each one can be run by itself.

**Objective = Revenue − Penalty − Shipping cost**

## 1. Setup

This notebook runs on its own. It reads the data files, does its own work, and prints the results.
It does not upload anything, and it does not save any files.

Set `DATA_DIR` below to the folder that holds the CSV files. If the folder is somewhere under the
current directory, the notebook finds it by itself.

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [3]:
import os, glob, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

# Put the path to your data folder here, or leave it as None to search for it.
DATA_DIR = "/content/drive/MyDrive/DOM-data"

def find_data(start=DATA_DIR):
    """Return the folder that holds input_order data.csv."""
    if start and os.path.isfile(os.path.join(start, "input_order data.csv")):
        return start
    for hit in glob.glob("**/input_order data.csv", recursive=True):
        return os.path.dirname(hit)          # first match wins
    raise FileNotFoundError(
        "Could not find 'input_order data.csv'. Set DATA_DIR to the folder that holds the CSV files.")

IN = find_data()                              # folder with the 5 input files
ROOT = os.path.dirname(IN) if os.path.basename(IN) == "input data" else IN
print("input folder :", IN)

input folder : drive/MyDrive/DOM/dom_extract/Nestle - WISER WQ26 DOM-data [SHARED]/DOM-data/input data


## 2. Read the data

Five input files. Two output files come from the earlier model run; we use them only to check our
own numbers, never as input.

In [4]:
orders = pd.read_csv(f"{IN}/input_order data.csv")            # one row per order and SKU
cap    = pd.read_csv(f"{IN}/input_capacity_planning.csv")     # daily stock per DC and SKU
dock   = pd.read_csv(f"{IN}/input_dock_capacity.csv")         # daily dock slots per DC
ship   = pd.read_csv(f"{IN}/input_shipping_cost_data.csv")    # cost per DC to zip lane
thru   = pd.read_csv(f"{IN}/input_throughput_capacity.csv")   # daily pick work per DC

def read_output(name):
    """The two output files may sit beside the input folder or one level up."""
    for p in [f"{ROOT}/{name}", f"{IN}/{name}", f"{ROOT}/../{name}"]:
        if os.path.isfile(p):
            return pd.read_csv(p)
    return None

poc_ord = read_output("Output_order_level_data.csv")          # check only
poc_sku = read_output("output_order_sku_level_data.csv")      # check only

for n, d in [("orders", orders), ("capacity", cap), ("dock", dock),
             ("shipping", ship), ("throughput", thru)]:
    print(f"{n:11s} {d.shape}")

orders      (25193, 39)
capacity    (377504, 23)
dock        (480, 13)
shipping    (12922, 7)
throughput  (530, 7)


## 3. Prepare the orders

Four small steps: parse the dates, put every line into cases, work out a price per case, and roll the
lines up to one row per order.

In [5]:
orders["PGI"] = pd.to_datetime(orders["transportationplanningdate"], format="%m/%d/%y")  # ship day
orders["RDD"] = pd.to_datetime(orders["RequestedDeliveryDate"],      format="%m/%d/%y")  # due day
orders["IsInvAvail"] = orders["IsInvAvail"].astype(str).str.strip().str.upper()

# Lines measured in pallets become cases, so everything is in the same unit.
orders["cases"] = np.where(orders["ProductPlanningUnitOfMeasure"].eq("PL"),
                           orders["OrderedQty_converted"] * orders["ProductCasesPerPallet"],
                           orders["OrderedQty_converted"]).astype(float)

# Price per case lets us rebuild revenue from the cases we actually fill.
orders["price_per_case"] = np.where(orders["cases"] > 0,
                                    orders["Order_SKU_Revenue"] / orders["cases"], 0.0)
orders["cpp"] = orders["ProductCasesPerPallet"].replace(0, np.nan).fillna(1).astype(float)

# One row per order.
head = orders.groupby("Group_Flag").agg(
    default_dc=("Plant", "first"),            # the DC it uses today
    pgi=("PGI", "first"),                     # planned ship day
    rdd=("RDD", "first"),                     # customer due day
    zipc=("ZipCode", "first"),                # ship-to zip, for the freight cost
    prio=("DeliveryPriority", "min"),         # 8 = protected order, 99 = normal
    revenue=("Order_SKU_Revenue", "sum"),
    ordered_cases=("cases", "sum"),
    frt=("FillRateThreshold", "first"),       # fill % below which a penalty starts
    ppc=("Penaltyforpotentialcuts", "first"), # penalty rate on the missing value
).reset_index()
head[["frt", "ppc"]] = head[["frt", "ppc"]].fillna(0.0)
head["threshold_cases"] = head["frt"] * head["ordered_cases"]

HEAD  = head.set_index("Group_Flag").to_dict("index")
LINES = {gf: [(int(r.MaterialNumber), float(r.cases), float(r.price_per_case), float(r.cpp))
              for r in g.itertuples(index=False)]
         for gf, g in orders.groupby("Group_Flag", sort=False)}

print("orders:", len(head), "| SKUs:", orders.MaterialNumber.nunique(),
      "| DCs:", orders.Plant.nunique())

orders: 1109 | SKUs: 1110 | DCs: 8


## 4. Find the focus orders

An order needs a decision for one of two reasons. Either it is short of stock at its own DC, or it
ships on a day when that DC has no dock slot left. Everything else is fine and stays where it is.

In [6]:
# Short of stock: any line of the order has IsInvAvail = N.
roll   = orders.groupby("Group_Flag")["IsInvAvail"].apply(lambda s: (s == "Y").all())
SUFF   = set(roll[roll].index)
INSUFF = set(roll[~roll].index)

# No dock left on the planned ship day.
dock["Date"] = pd.to_datetime(dock["Date"], format="%m/%d/%y")
zero_days = set(map(tuple, dock.loc[dock["Dock_Remaining"] == 0, ["Plant", "Date"]]
                              .drop_duplicates().values))
ADDED = {gf for gf, h in HEAD.items()
         if (h["default_dc"], h["pgi"]) in zero_days} & SUFF

FOCUS = sorted(INSUFF | ADDED)                        # the orders the model decides
CLEAN_ORDERS = sorted(set(HEAD) - set(FOCUS))         # the orders that stay put

print(f"{len(INSUFF)} short of stock + {len(ADDED)} with no dock = {len(FOCUS)} focus orders")
print(f"{len(CLEAN_ORDERS)} orders are fine and stay where they are")

447 short of stock + 25 with no dock = 472 focus orders
637 orders are fine and stay where they are


## 5. Build the stock, dock and capacity tables

Everything that changes by day sits on the same day grid, so "the next 5 days" is just a slice of an
array.

In [7]:
cap["DATE"] = pd.to_datetime(cap["DATE"])
DATES = pd.date_range(cap["DATE"].min(), cap["DATE"].max(), freq="D")
NT    = len(DATES)                                    # number of days
DIX   = {d: i for i, d in enumerate(DATES)}           # date -> position
DCS   = sorted(int(x) for x in orders["Plant"].unique())
DCIX  = {d: i for i, d in enumerate(DCS)}             # DC -> row number

# Free stock = opening stock minus what is already reserved.
cap["AV"] = cap["OpeningStock"] - cap["Total_Reserved_Qty"]
POOL0 = {}
for (d, s), g in cap[cap["LocationID"].isin(DCS)].groupby(["LocationID", "MaterialID"], sort=False):
    arr = np.zeros(NT)
    ix  = g["DATE"].map(DIX).dropna().astype(int).values
    arr[ix] = g["AV"].values[:len(ix)]
    POOL0[(int(d), int(s))] = arr                     # one stock line per DC and SKU

SKU_AT_DC = {}
for (d, s) in POOL0:
    SKU_AT_DC.setdefault(d, set()).add(s)             # which DC stocks which SKU

# Freight cost and distance for each DC to zip lane.
SHIP = {(int(r.Plant), int(r.TargetZip)): float(r.Shipping_Cost)
        for r in ship.itertuples(index=False)}
DIST = {(int(r.Plant), int(r.TargetZip)): float(r.Distance)
        for r in ship.itertuples(index=False)}

# Free dock slots. Two DCs have no rows at all, so we mark them "no data" and skip the check there.
DOCK0 = np.full((len(DCS), NT), np.nan)
for r in dock.itertuples(index=False):
    if r.Plant in DCIX and r.Date in DIX:
        DOCK0[DCIX[r.Plant], DIX[r.Date]] = max(0.0, float(r.Dock_Remaining))
DOCK_HAS = ~np.isnan(DOCK0)
DOCK0    = np.nan_to_num(DOCK0, nan=0.0)

# Pick capacity. The file gives usage only, so we take the highest usage as the limit.
thru["transportationplanningdate"] = pd.to_datetime(thru["transportationplanningdate"])
CP_CAP = thru.groupby("Plant")["util_case_picks"].max().to_dict()
PP_CAP = thru.groupby("Plant")["util_pallets"].max().to_dict()
CP0 = np.zeros((len(DCS), NT)); PP0 = np.zeros((len(DCS), NT))
for d in DCS:
    CP0[DCIX[d], :] = CP_CAP.get(d, 0.0)
    PP0[DCIX[d], :] = PP_CAP.get(d, 0.0)
for r in thru.itertuples(index=False):
    if r.Plant in DCIX and r.transportationplanningdate in DIX:
        i, j = DCIX[r.Plant], DIX[r.transportationplanningdate]
        CP0[i, j] = max(0.0, CP_CAP[r.Plant] - r.util_case_picks)
        PP0[i, j] = max(0.0, PP_CAP[r.Plant] - r.util_pallets)

print("days:", NT, "| DCs:", DCS, "| DC-SKU stock lines:", len(POOL0))

days: 32 | DCs: [5083, 5385, 5410, 5420, 5490, 5620, 5641, 5773] | DC-SKU stock lines: 11550


## 6. The rules and the objective

The business rules from the project documents, written as small functions.

**Objective = Revenue − Penalty − Shipping cost**

| Rule | What it means |
|---|---|
| C1 | One order goes to one DC only |
| C2 | You cannot fill more than what was ordered |
| C3 | Stock must be there, and must still cover the next 5 days |
| C4 | Move only if the fill rises 5 points **and** 100 cases |
| C5 | Case picks and pallet picks must fit the DC limit that day |
| C6 | One dock slot per order, and docks are limited |
| C7 | A penalty applies only if filled cases fall under the order threshold |

In [8]:
CFG = dict(
    MIN_FILL_LIFT_PP   = 0.05,   # C4: the fill must rise 5 points
    MIN_CASE_LIFT      = 100.0,  # C4: and at least 100 cases
    FORWARD_COVER_DAYS = 5,      # C3: stock must last 5 more days
    LEAD_TIME_MILES    = 500.0,  # one transit day per 500 miles
    DOCKS_PER_ORDER    = 1,      # C6: one dock per order
    SAFETY_STOCK_FRAC  = 0.0,    # hold back stock elsewhere (sensitivity only)
    REQUIRE_OBJ_GAIN   = True,   # greedy only: move only if the money improves
)

def avail(P, d, s, t, window=0):
    """C3: how many cases we may take. With a window, the tightest day decides."""
    a = P.get((d, s))
    if a is None:
        return 0.0
    hi = min(NT, t + window + 1)
    return max(0.0, a[t:hi].min()) if hi > t else 0.0

def take(P, d, s, t, q):
    a = P.get((d, s))
    if a is not None and q > 0:
        a[t:] -= q                                  # gone from today and every later day

def give(P, d, s, t, q):
    a = P.get((d, s))
    if a is not None and q > 0:
        a[t:] += q                                  # put it back the same way

def evaluate(P, gf, d, t, window=0, reserve=0.0):
    """C2 and C3: how much of this order this DC can fill on this day."""
    fills, by, tot, rev = [], {}, 0.0, 0.0
    for s, dem, price, cpp in LINES[gf]:
        q = min(dem, avail(P, d, s, t, window) * (1.0 - reserve))   # never more than ordered
        fills.append((s, q, price, cpp)); by[s] = q; tot += q; rev += q * price
    return fills, by, tot, rev

def picks(fills):
    """C5: full pallets are pallet picks, the rest are case picks."""
    cp = pp = 0.0
    for _, q, _, cpp in fills:
        if q <= 0:
            continue
        full = math.floor(q / cpp); pp += full; cp += q - full * cpp
    return cp, pp

def penalty_of(gf, by, tot):
    """C7: no penalty if we reach the threshold, otherwise charge for what is missing."""
    h = HEAD[gf]
    if h["ppc"] <= 0 or tot >= h["threshold_cases"]:
        return 0.0
    return max(0.0, sum((dem - by.get(s, 0.0)) * price * h["ppc"]
                        for s, dem, price, _ in LINES[gf]))

def objective(rec):
    """Revenue minus penalty minus shipping."""
    return rec["revenue"] - rec["pen"] - rec["ship"]

def lead_time(d, z):
    dd = DIST.get((d, z))
    return None if dd is None else max(1, int(math.ceil(dd / CFG["LEAD_TIME_MILES"])))

def revised_pgi(gf, d):
    """A farther DC needs more transit time, so the ship day moves earlier."""
    h = HEAD[gf]; lt = lead_time(d, h["zipc"])
    if lt is None:
        return None, "no_lane"
    p = min(h["pgi"], h["rdd"] - pd.Timedelta(days=lt))
    while p.weekday() >= 5:                          # step back off Saturday and Sunday
        p -= pd.Timedelta(days=1)
    return (None, "pgi_out_of_horizon") if p not in DIX else (p, None)

def passes_gate(lift, ordered, cfg=CFG):
    """C4: the move must be worth making."""
    if lift < cfg["MIN_FILL_LIFT_PP"] * ordered:
        return "fail_5pct"
    if lift < cfg["MIN_CASE_LIFT"]:
        return "fail_100cases"
    return None

print("rules ready")

rules ready


## 7. Put every order at its own DC

This is the starting point. Protected orders (priority 8) go first, then the largest by revenue.
Each order takes its stock, so the next order sees what is left. This is also **Baseline 1**.

In [9]:
def stage_A():
    P = {k: v.copy() for k, v in POOL0.items()}      # work on a copy of the stock
    seq = head.sort_values(["prio", "revenue"], ascending=[True, False])["Group_Flag"]
    out = {}
    for gf in seq:
        h = HEAD[gf]; d = h["default_dc"]; t = DIX[h["pgi"]]
        fills, by, tot, rev = evaluate(P, gf, d, t, window=0)
        for s, q, _, _ in fills:
            take(P, d, s, t, q)                      # remove that stock for good
        cp, pp = picks(fills)
        out[gf] = dict(dc=d, t=t, fills=fills, by=by, filled=tot, revenue=rev, cp=cp, pp=pp,
                       pen=penalty_of(gf, by, tot), ship=SHIP.get((d, h["zipc"]), 0.0),
                       cof=tot / h["ordered_cases"] if h["ordered_cases"] else 0.0)
    return P, out

POOL_A, DEF = stage_A()
print("done:", len(DEF), "orders placed")

done: 1109 orders placed


## 8. Check the preparation

Two checks. The first shows what the starting point is worth. The second compares our fill rate with
the earlier model output. Ordered cases should match exactly. Fill rate should be close but not the
same, because the earlier run held stock back for a demand forecast that is not in these files.

In [10]:
oc = sum(HEAD[g]["ordered_cases"] for g in FOCUS)
fl = sum(DEF[g]["filled"] for g in FOCUS)
print("focus orders     :", len(FOCUS))
print("objective        :", f"{sum(objective(DEF[g]) for g in FOCUS):,.0f}")
print("fill rate        :", f"{fl / oc * 100:.2f}%")
print("cases ordered    :", f"{oc:,.0f}")
print("cases filled     :", f"{fl:,.0f}")

focus orders     : 472
objective        : 44,365,994
fill rate        : 90.47%
cases ordered    : 1,449,768
cases filled     : 1,311,570


In [11]:
if poc_ord is not None:
    po = poc_ord.set_index("SalesDocument/GroupingIndicator")
    V = pd.DataFrame([(DEF[g]["cof"], float(po.loc[g, "Default_DC_COF"]),
                       HEAD[g]["ordered_cases"], float(po.loc[g, "OrderedQty_Cases"]))
                      for g in DEF if g in po.index],
                     columns=["cof_ours", "cof_poc", "cases_ours", "cases_poc"])
    print("ordered cases match   :", round((V.cases_ours == V.cases_poc).mean(), 4))
    print("fill rate within 5 pts:", round((abs(V.cof_ours - V.cof_poc) < 0.05).mean(), 4))
    print("mean fill ours / theirs:", round(V.cof_ours.mean(), 4), "/", round(V.cof_poc.mean(), 4))

ordered cases match   : 1.0
fill rate within 5 pts: 0.8638
mean fill ours / theirs: 0.895 / 0.9422


## 9. Assumptions

Each one exists because something is missing from the files, not because it was easier.

| # | We assumed | Why |
|---|---|---|
| 1 | Focus orders = short of stock **plus** no dock that day | no throughput limit in the files |
| 2 | Free stock = opening stock − reserved | exact on every row |
| 3 | Transit days = distance ÷ 500, rounded up | matches the earlier output on every row |
| 4 | Pick limit = the highest usage ever seen | the file gives usage, never a limit |
| 5 | Holidays = weekends only | no holiday file |
| 6 | "SKU is in the forecast" = the DC holds stock rows for it | no forecast file |
| 7 | The gate is 0.05, not 1.05 | 1.05 is impossible; the text says 5% |
| 8 | Minimum and maximum penalty are not used | 0 means "not set", not "capped at zero" |
| 9 | No dock check at DC 5083 and 5773 | they have no dock rows |
| 10 | No stock held back for a forecast | not in the files |